<a href="https://colab.research.google.com/github/nishithy13/DS2002-Midterm/blob/main/06-midterm-walmart/2026-10-23%20%E2%80%94%20Midterm%20Checkpoint%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [57]:
team_setup = {
    'team_name': 'Iceman',
    'members': [                                  # everyone on the team, 2 to 4 people
        {'name': 'Nishith Yalamanchi', 'github': 'nishithy13'},
        {'name': 'Sid Luthra', 'github': 'sidluthra1'},
        {'name': 'Ishan Alla', 'github': 'ishanallasanagala'},

    ],
    'repo_url': 'https://github.com/nishithy13/DS2002-Midterm/tree/main',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/nishithy13/DS2002-Midterm/blob/main/CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Ishan Alla',           # nice to have
    'meeting_time': 'Mon and Wed 9 am',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': [
        'unit_price is stored as text and some values start with $',
        'quantity contains negative numbers that look like returns and have missing values'
        'timestamp uses mixed string formats and has some unparseable/missing rows',
        'the same product is fragmented across multiple SKUs and names such as Strawberry Pop-Tarts',
        'store_id uses inconsistent casing and separators across files such as fl-239, FL_239, FL239',
    ],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  not yet - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [58]:
import os
import subprocess

REPO_URL = 'https://github.com/nishithy13/DS2002-Midterm.git'
REPO_PATH = 'DS2002-Midterm'  # local clone of the team repo

if not os.path.isdir(REPO_PATH):
    subprocess.run(['git', 'clone', REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(['git', '-C', REPO_PATH, 'pull'], check=True)

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

b1a1c05 Update CHARTER.md with Nishith Yalamanchi and revised Q1/Q2 owners


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [59]:
# TODO
import os
import pandas as pd

def data_path(filename):
    for folder in ("data", "../data", "06-midterm-walmart/data", "DS2002-Midterm/data", "DS2002-Midterm/06-midterm-walmart/data", "."):
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return os.path.join("data", filename)

def drop_duplicates(df):
    return df.drop_duplicates()

def drop_negative_quantity(df):
    # Team decision: negatives look like returns, not sales. Blank quantities are kept.
    qty = pd.to_numeric(df["quantity"], errors="coerce")
    return df.loc[~(qty < 0)]

def clean_unit_price(df):
    df = df.copy()
    stripped = df["unit_price"].astype("string").str.replace(r"[$,]", "", regex=True)
    df["unit_price"] = pd.to_numeric(stripped, errors="coerce")
    return df

def parse_timestamp(df):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="mixed", errors="coerce")
    return df

def clean_store_id(df):
    df = df.copy()
    df["store_id"] = (df["store_id"].astype("string").str.strip().str.upper()
                      .str.replace("_", "-", regex=False)
                      .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
                      .replace("", pd.NA))
    return df

def drop_unusable(df):
    return df.dropna(subset=["unit_price", "timestamp", "store_id"])

PIPELINE = [drop_duplicates, drop_negative_quantity, clean_unit_price,
            parse_timestamp, clean_store_id, drop_unusable]

def clean_data(raw):
    df = raw
    for step in PIPELINE:
        df = step(df)
    return df

original_df = pd.read_csv(data_path("walmart_transactions.csv"))
clean_df = clean_data(original_df)
print("Rows before cleaning:", len(original_df))
print("Rows after cleaning:", len(clean_df))
print("Rows removed:", len(original_df) - len(clean_df))

Rows before cleaning: 25562
Rows after cleaning: 24521
Rows removed: 1041


### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [60]:
# TODO: print or display your decision log
import pandas as pd
from IPython.display import display

def clean_with_log(raw):
    df = raw.copy()
    log = []

    def record(decision, touched, removed=0):
        log.append({
            "decision": decision,
            "rows_touched": int(touched),
            "rows_removed": int(removed)
        })

    # Remove exact duplicate rows.
    duplicates = df.duplicated()
    record("Remove exact duplicate rows", duplicates.sum(), duplicates.sum())
    df = df.loc[~duplicates].copy()

    # Keep the team's decision to exclude negative quantities.
    negative = df["quantity"] < 0
    record(
        "Exclude negative quantities; possible returns, not positive sales",
        negative.sum(), negative.sum()
    )
    df = df.loc[~negative].copy()

    # Remove currency formatting and convert prices to numbers.
    prices = df["unit_price"].astype("string")
    stripped = prices.str.replace(r"[$,]", "", regex=True)
    record("Remove $ and commas from prices", prices.ne(stripped).sum())
    df["unit_price"] = pd.to_numeric(stripped, errors="coerce")
    record("Convert prices to numeric", df["unit_price"].notna().sum())

    # Mixed formats are valid; do not discard them accidentally.
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="mixed", errors="coerce"
    )
    record(
        "Parse mixed timestamp formats; invalid values become NaT",
        len(df)
    )

    # Standardize IDs so FL239, fl-239, and FL_239 become FL-239.
    old_ids = df["store_id"].astype("string")
    df["store_id"] = (
        old_ids.str.strip()
        .str.upper()
        .str.replace("_", "-", regex=False)
        .str.replace(r"^FL(\d+)$", r"FL-\1", regex=True)
        .replace("", pd.NA)
    )
    record("Standardize store IDs for joins", old_ids.ne(df["store_id"]).sum())

    invalid = df[["unit_price", "timestamp", "store_id"]].isna().any(axis=1)
    record(
        "Drop rows missing a usable price, timestamp, or store ID",
        invalid.sum(), invalid.sum()
    )
    df = df.loc[~invalid].copy()

    return df, pd.DataFrame(log)

original_df = pd.read_csv(data_path("walmart_transactions.csv"))
clean_df, decision_log = clean_with_log(original_df)

display(decision_log)
print("Rows before cleaning:", len(original_df))
print("Rows after cleaning:", len(clean_df))
print("Rows removed:", len(original_df) - len(clean_df))
print("Rows touched can overlap across decisions.")
print("Team disagreements: not yet documented; confirm with teammates.")

,decision,rows_touched,rows_removed
0,Remove exact duplicate rows,30,30
1,"Exclude negative quantities; possible returns,...",130,130
2,Remove $ and commas from prices,3016,0
3,Convert prices to numeric,25402,0
4,Parse mixed timestamp formats; invalid values ...,25402,0
5,Standardize store IDs for joins,4324,0
6,"Drop rows missing a usable price, timestamp, o...",881,881


Rows before cleaning: 25562
Rows after cleaning: 24521
Rows removed: 1041
Rows touched can overlap across decisions.
Team disagreements: not yet documented; confirm with teammates.


### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [61]:
# TODO
catalog = pd.read_csv(data_path("product_catalog.csv"))
catalog["sku"] = catalog["sku"].astype("string").str.strip().str.upper()

pop_tarts_skus = [
    "PT-12",
    "SKU#459812",
    "POPTART-STRAW",
    "PT_STRAWBERRY"
]

# Include every SKU from both the transactions and catalog.
all_skus = pd.concat([
    original_df["sku"].astype("string").str.strip().str.upper(),
    catalog["sku"]
]).dropna().drop_duplicates()

sku_mapping = pd.DataFrame({"sku": sorted(all_skus)})
sku_mapping["canonical_product"] = sku_mapping["sku"]
sku_mapping["status"] = "Not reviewed; kept separate"

is_pop_tarts = sku_mapping["sku"].isin(pop_tarts_skus)
sku_mapping.loc[
    is_pop_tarts, "canonical_product"
] = "Strawberry Pop-Tarts"
sku_mapping.loc[
    is_pop_tarts, "status"
] = "Provisional product-family match"

group_sizes = (
    sku_mapping.groupby("canonical_product")["sku"]
    .nunique()
    .sort_values(ascending=False)
)

print("Distinct starting SKUs:", original_df["sku"].nunique())
print("Current canonical groups:", sku_mapping["canonical_product"].nunique())
print("Largest consolidated group:", group_sizes.index[0])
print("SKUs in largest group:", group_sizes.iloc[0])

family_mapping = catalog.merge(
    sku_mapping, on="sku", how="left", validate="many_to_one"
)
display(
    family_mapping.loc[
        family_mapping["sku"].isin(pop_tarts_skus),
        ["sku", "product_name", "package_size", "canonical_product", "status"]
    ]
)

clean_df["sku"] = clean_df["sku"].astype("string").str.strip().str.upper()
clean_df = clean_df.drop(
    columns=["canonical_product"], errors="ignore"
).merge(
    sku_mapping[["sku", "canonical_product"]],
    on="sku", how="left", validate="many_to_one"
)

print(
    "Reason: all four catalog names describe strawberry Pop-Tarts. "
    "Package sizes conflict, so this is a provisional family grouping, "
    "not proof that the packages contain equivalent quantities."
)

Distinct starting SKUs: 45
Current canonical groups: 42
Largest consolidated group: Strawberry Pop-Tarts
SKUs in largest group: 4


,sku,product_name,package_size,canonical_product,status
0,PT-12,pop-tarts strawberry,4-pack,Strawberry Pop-Tarts,Provisional product-family match
1,SKU#459812,Strawberry Pop-Tart,4pk,Strawberry Pop-Tarts,Provisional product-family match
2,POPTART-STRAW,Pop-Tarts Strawberry,6 pack,Strawberry Pop-Tarts,Provisional product-family match
3,PT_STRAWBERRY,pop-tarts strawberry,4 pack,Strawberry Pop-Tarts,Provisional product-family match


Reason: all four catalog names describe strawberry Pop-Tarts. Package sizes conflict, so this is a provisional family grouping, not proof that the packages contain equivalent quantities.


### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [62]:
# TODO
import json
import time
import urllib.request
import urllib.parse
import urllib.error

def get_weather(latitude, longitude, start_date, end_date):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation,wind_speed_10m",
        "timezone": "UTC"
    }
    url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        + urllib.parse.urlencode(params)
    )
    print("API call:", url)

    for attempt in range(3):
        try:
            with urllib.request.urlopen(url, timeout=30) as response:
                data = json.load(response)

            weather = pd.DataFrame(data["hourly"]).rename(
                columns={"time": "weather_hour_utc"}
            )
            weather["weather_hour_utc"] = pd.to_datetime(
                weather["weather_hour_utc"], utc=True
            )
            weather["source"] = "Open-Meteo Historical Weather API"
            return weather

        except (
            urllib.error.URLError, TimeoutError, OSError,
            ValueError, KeyError
        ) as error:
            print(f"Attempt {attempt + 1} failed: {error}")
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)

stores = pd.read_csv(data_path("store_locations.csv"))
stores["store_id"] = stores["store_id"].str.strip().str.upper()
orlando = stores.loc[stores["store_id"] == "FL-239"].iloc[0]

weather_df = get_weather(
    latitude=float(orlando["latitude"]),
    longitude=float(orlando["longitude"]),
    start_date="2024-09-01",
    end_date="2024-10-01"
)
weather_df["store_id"] = "FL-239"

display(weather_df.head())
print("Weather rows:", len(weather_df))
print("Units: temperature = °C, precipitation = mm, wind speed = km/h")

# Assumption: Orlando sales timestamps represent local store time.
# Localize them first, then convert to UTC for the hourly join.
orlando_sales = clean_df.loc[clean_df["store_id"] == "FL-239"].copy()
orlando_sales["weather_hour_utc"] = (
    orlando_sales["timestamp"]
    .dt.tz_localize("America/New_York")
    .dt.tz_convert("UTC")
    .dt.floor("h")
)

sales_weather = orlando_sales.merge(
    weather_df,
    on=["store_id", "weather_hour_utc"],
    how="left",
    validate="many_to_one"
)

print("Weather timezone: UTC.")
print(
    "Sales timezone assumption: America/New_York for Orlando; "
    "confirm with the data provider."
)
print("Sales rows with weather:", sales_weather["source"].notna().sum())
display(sales_weather.head())

API call: https://archive-api.open-meteo.com/v1/archive?latitude=28.5383&longitude=-81.3792&start_date=2024-09-01&end_date=2024-10-01&hourly=temperature_2m%2Cprecipitation%2Cwind_speed_10m&timezone=UTC


,weather_hour_utc,temperature_2m,precipitation,wind_speed_10m,source,store_id
0,2024-09-01 00:00:00+00:00,26.8,0.0,7.2,Open-Meteo Historical Weather API,FL-239
1,2024-09-01 01:00:00+00:00,25.9,0.0,6.8,Open-Meteo Historical Weather API,FL-239
2,2024-09-01 02:00:00+00:00,25.7,0.0,10.3,Open-Meteo Historical Weather API,FL-239
3,2024-09-01 03:00:00+00:00,25.2,0.0,8.9,Open-Meteo Historical Weather API,FL-239
4,2024-09-01 04:00:00+00:00,24.8,0.0,7.9,Open-Meteo Historical Weather API,FL-239


Weather rows: 744
Units: temperature = °C, precipitation = mm, wind speed = km/h
Weather timezone: UTC.
Sales timezone assumption: America/New_York for Orlando; confirm with the data provider.
Sales rows with weather: 2010


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,canonical_product,weather_hour_utc,temperature_2m,precipitation,wind_speed_10m,source
0,TXN005685,FL-239,2024-09-20 07:43:43,BEER-24,Light Beer 24pk,ALCOHOL,3,18.84,BEER-24,2024-09-20 11:00:00+00:00,22.3,0.0,5.8,Open-Meteo Historical Weather API
1,TXN012742,FL-239,2024-09-25 14:21:48,BATT-100,BATTERIES AA 4PK,Emergency Supplies,5,6.44,BATT-100,2024-09-25 18:00:00+00:00,27.4,0.1,25.7,Open-Meteo Historical Weather API
2,TXN002411,FL-239,2024-09-16 17:26:07,SOUP-CAN,CANNED SOUP,Groceries,4,1.92,SOUP-CAN,2024-09-16 21:00:00+00:00,25.8,0.3,10.3,Open-Meteo Historical Weather API
3,TXN005696,FL-239,2024-09-20 12:59:00,SOUP-CAN,Soup - Canned,GROCERY,3,1.98,SOUP-CAN,2024-09-20 16:00:00+00:00,29.2,0.1,12.4,Open-Meteo Historical Weather API
4,TXN023724,FL-239,2024-09-09 08:12:28,CNDL-EM,CANDLES - EMERGENCY,EMERGENCY,3,6.05,CNDL-EM,2024-09-09 12:00:00+00:00,24.0,0.0,5.6,Open-Meteo Historical Weather API


### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [63]:
# not started

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [64]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': int(len(clean_df)),                          # an integer
    'decisions_logged': int(len(decision_log)),                         # how many entries in your log
    'canonical_products': int(sku_mapping["canonical_product"].nunique()),  # distinct products after consolidation
    'weather_api_rows': int(len(weather_df)),                           # rows returned by your API call
    'notebook_runs_clean': True,                                        # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  done - rows_after_cleaning = 24521
  done - decisions_logged = 7
  done - canonical_products = 42
  done - weather_api_rows = 744
  done - notebook_runs_clean = True

required items complete: True
progress items done: 5 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** Extending the provisional SKU consolidation table beyond Strawberry Pop-Tarts to all 45 catalog products and resolving package-size mismatches before running Q2 and Q5.

**Plan, with an owner and a date:** Sid Luthra and Ishan Alla will review all remaining unmapped SKUs in `product_catalog.csv` and confirm/lock the finalized canonical mapping table by Sunday, November 1st.

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.

| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | completed | Sid Luthra, Ishan Alla, and Nishith Yalamanchi |
| Q2 SKU consolidation | completed | Sid Luthra and Ishan Alla, |
| Q3 weather correlation | completed | Ishan Alla and Sid Luthra |
| Q4 store and geography | not started | N/A |
| Q5 signal or artifact | not started | N/A |

At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.